# 📊 Phân Tích Khám Phá Dữ Liệu (EDA) — CyberShield AI

**Dự án:** Phát hiện website lừa đảo (Phishing Detection)  
**Bộ dữ liệu:** UCI Phishing Websites Dataset  
**Tác giả:** CyberShield AI Team  

---

Notebook này thực hiện phân tích khám phá dữ liệu (Exploratory Data Analysis) toàn diện cho bộ dữ liệu
phishing websites từ UCI Machine Learning Repository. Mục tiêu:

1. Hiểu cấu trúc và phân phối dữ liệu
2. Phân tích mối tương quan giữa các đặc trưng
3. Xác định các đặc trưng quan trọng nhất cho việc phân loại
4. Phát hiện đa cộng tuyến tiềm ẩn
5. Đưa ra khuyến nghị cho giai đoạn huấn luyện mô hình

## 1. 🔧 Cài đặt & Tải dữ liệu

Import các thư viện cần thiết và sử dụng hạ tầng `cybershield_ai` có sẵn để tải dữ liệu UCI.

In [ ]:
# === Cài đặt môi trường ===
%matplotlib inline

import sys
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Thêm thư mục gốc dự án vào sys.path để import cybershield_ai
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Import từ hạ tầng dự án
from cybershield_ai.data_loader import load_uci_dataset
from cybershield_ai.config import FEATURE_COLUMNS, RESULT_COLUMN

# Cấu hình hiển thị
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 35)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 180
plt.rcParams["savefig.bbox"] = "tight"

# Tạo thư mục lưu kết quả
REPORT_DIR = os.path.join(PROJECT_ROOT, "artifacts", "reports")
os.makedirs(REPORT_DIR, exist_ok=True)

print(f"✅ Thư mục dự án: {PROJECT_ROOT}")
print(f"✅ Thư mục báo cáo: {REPORT_DIR}")

In [ ]:
# === Tải dữ liệu UCI Phishing Websites ===
dataset = load_uci_dataset()

df = dataset.dataframe     # DataFrame gốc (30 features + Result)
X = dataset.X              # Ma trận đặc trưng (30 cột)
y = dataset.y              # Nhãn nhị phân: is_phishing (0 = hợp lệ, 1 = lừa đảo)
raw_target = dataset.raw_target  # Nhãn gốc: Result (-1 = phishing, 1 = legitimate)

print(f"📐 Kích thước DataFrame:  {df.shape}")
print(f"📐 Kích thước X (features): {X.shape}")
print(f"📐 Kích thước y (target):   {y.shape}")
print(f"\n📋 Danh sách {len(FEATURE_COLUMNS)} đặc trưng:")
for i, col in enumerate(FEATURE_COLUMNS, 1):
    print(f"   {i:2d}. {col}")

In [ ]:
# === Xem trước dữ liệu ===
print("🔍 5 dòng đầu tiên:")
df.head()

In [ ]:
# === Thông tin kiểu dữ liệu ===
print("📊 Thông tin DataFrame:")
df.info()

In [ ]:
# === Thống kê mô tả ===
print("📈 Thống kê mô tả:")
df.describe()

---
## 2. ⚖️ Phân phối nhãn (Class Distribution)

Kiểm tra tỷ lệ giữa website hợp lệ và website lừa đảo. Đây là bước quan trọng để đánh giá xem
bộ dữ liệu có bị **mất cân bằng (imbalanced)** hay không, từ đó quyết định chiến lược huấn luyện.

In [ ]:
# === Phân phối nhãn gốc (Result: -1 = phishing, 1 = legitimate) ===
print("📊 Phân phối nhãn GỐC (Result):")
print(raw_target.value_counts().to_string())
print()

# === Phân phối nhãn đã chuyển đổi (is_phishing: 0 = hợp lệ, 1 = lừa đảo) ===
print("📊 Phân phối nhãn ĐÃ CHUYỂN ĐỔI (is_phishing):")
print(y.value_counts().to_string())

In [ ]:
# === Biểu đồ phân phối nhãn ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bảng màu tùy chỉnh
colors_raw = ["#e74c3c", "#2ecc71"]   # đỏ = phishing(-1), xanh = legitimate(1)
colors_bin = ["#2ecc71", "#e74c3c"]   # xanh = legitimate(0), đỏ = phishing(1)

# --- Panel trái: nhãn gốc ---
counts_raw = raw_target.value_counts().sort_index()
bars1 = axes[0].bar(counts_raw.index.astype(str), counts_raw.values, color=colors_raw,
                    edgecolor="white", linewidth=1.5, width=0.5)
axes[0].set_title("Phân phối nhãn GỐC (Result)", fontsize=14, fontweight="bold", pad=12)
axes[0].set_xlabel("Giá trị Result", fontsize=12)
axes[0].set_ylabel("Số lượng", fontsize=12)
total_raw = counts_raw.sum()
for bar, val in zip(bars1, counts_raw.values):
    pct = val / total_raw * 100
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f"{val:,}\n({pct:.1f}%)", ha="center", va="bottom", fontsize=11, fontweight="bold")

# --- Panel phải: nhãn nhị phân ---
counts_bin = y.value_counts().sort_index()
labels_bin = ["Hợp lệ (0)", "Lừa đảo (1)"]
bars2 = axes[1].bar(labels_bin, counts_bin.values, color=colors_bin,
                    edgecolor="white", linewidth=1.5, width=0.5)
axes[1].set_title("Phân phối nhãn is_phishing", fontsize=14, fontweight="bold", pad=12)
axes[1].set_xlabel("Nhãn", fontsize=12)
axes[1].set_ylabel("Số lượng", fontsize=12)
total_bin = counts_bin.sum()
for bar, val in zip(bars2, counts_bin.values):
    pct = val / total_bin * 100
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f"{val:,}\n({pct:.1f}%)", ha="center", va="bottom", fontsize=11, fontweight="bold")

fig.suptitle("⚖️ Phân Phối Nhãn — UCI Phishing Websites Dataset",
             fontsize=16, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

### 💡 Nhận xét

- Bộ dữ liệu có tỷ lệ khá **cân bằng** giữa hai lớp (phishing vs legitimate).
- Số lượng mẫu phishing (Result = -1) và hợp lệ (Result = 1) xấp xỉ nhau.
- **Kết luận:** Không cần sử dụng các kỹ thuật xử lý mất cân bằng dữ liệu (SMOTE, undersampling, v.v.).

---
## 3. 🔎 Phân tích giá trị thiếu (Missing Values)

Kiểm tra xem bộ dữ liệu có giá trị thiếu (null/NaN) hay không.

In [ ]:
# === Kiểm tra giá trị thiếu ===
missing = df.isnull().sum()
total_missing = missing.sum()

print(f"📊 Tổng số giá trị thiếu trong toàn bộ DataFrame: {total_missing}")
print()

if total_missing > 0:
    # Nếu có giá trị thiếu → hiển thị chi tiết
    missing_cols = missing[missing > 0].sort_values(ascending=False)
    print("⚠️ Các cột có giá trị thiếu:")
    for col, count in missing_cols.items():
        pct = count / len(df) * 100
        print(f"   {col}: {count} ({pct:.2f}%)")
    
    # Visualize
    fig, ax = plt.subplots(figsize=(12, 5))
    missing_cols.plot(kind="barh", color="#e74c3c", ax=ax)
    ax.set_title("Số lượng giá trị thiếu theo cột", fontsize=14, fontweight="bold")
    ax.set_xlabel("Số lượng")
    plt.tight_layout()
    plt.show()
else:
    print("✅ Không có giá trị thiếu! Bộ dữ liệu đã được tiền xử lý sạch.")
    print(f"   Tổng số mẫu: {len(df):,}")
    print(f"   Tổng số cột:  {len(df.columns)}")
    print(f"   Tổng ô dữ liệu: {df.size:,}")

### 💡 Kết luận

- Bộ dữ liệu UCI Phishing Websites **không có giá trị thiếu**.
- Tất cả các đặc trưng đều là số nguyên rời rạc, đã được trích xuất sẵn từ URL và nội dung trang web.
- Không cần thực hiện bước imputation (điền giá trị thiếu).

---
## 4. 📊 Phân phối đặc trưng (Feature Distributions)

Tất cả 30 đặc trưng trong bộ dữ liệu đều là **biến phân loại** (categorical), với các giá trị
thuộc tập `{-1, 0, 1}` hoặc `{-1, 1}` hoặc `{0, 1}`. Ta sẽ trực quan hóa phân phối của từng đặc trưng.

In [ ]:
# === Phân phối 30 đặc trưng — Lưới 6x5 ===
n_features = len(FEATURE_COLUMNS)
n_cols = 5
n_rows = 6

palette = {-1: "#e74c3c", 0: "#f39c12", 1: "#2ecc71"}

fig, axes = plt.subplots(n_rows, n_cols, figsize=(22, 24))
fig.suptitle("📊 Phân Phối Giá Trị Của 30 Đặc Trưng",
             fontsize=20, fontweight="bold", y=1.01)

for idx, col in enumerate(FEATURE_COLUMNS):
    row, c = divmod(idx, n_cols)
    ax = axes[row][c]
    
    vc = X[col].value_counts().sort_index()
    bar_colors = [palette.get(v, "#95a5a6") for v in vc.index]
    
    bars = ax.bar(vc.index.astype(str), vc.values, color=bar_colors,
                  edgecolor="white", linewidth=1.2, width=0.5)
    
    ax.set_title(col, fontsize=10, fontweight="bold", pad=8)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(labelsize=9)
    
    # Ghi số lượng trên mỗi thanh
    total = vc.sum()
    for bar, val in zip(bars, vc.values):
        pct = val / total * 100
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f"{pct:.0f}%", ha="center", va="bottom", fontsize=8, fontweight="bold")

# Ẩn ô thừa nếu có
for idx in range(n_features, n_rows * n_cols):
    row, c = divmod(idx, n_cols)
    axes[row][c].set_visible(False)

plt.tight_layout()
save_path = os.path.join(REPORT_DIR, "eda_feature_distributions.png")
fig.savefig(save_path, dpi=180, bbox_inches="tight", facecolor="white")
print(f"💾 Đã lưu: {save_path}")
plt.show()

### 💡 Nhận xét

- Hầu hết các đặc trưng nhị phân `{-1, 1}` có phân phối khá đồng đều.
- Một số đặc trưng ba giá trị `{-1, 0, 1}` cho thấy sự thiên lệch rõ rệt.
- Các đặc trưng như `Redirect`, `RightClick`, `popUpWidnow`, `Iframe` có tỷ lệ giá trị 1 rất cao
  → có thể chỉ ra rằng phần lớn các trang web (cả hợp lệ lẫn lừa đảo) đều không có hành vi này.
- Đặc trưng `Redirect` có giá trị `{0, 1}` thay vì `{-1, 1}` như các đặc trưng khác.

---
## 5. 🔥 Ma trận tương quan (Correlation Matrix)

Tính và trực quan hóa ma trận tương quan giữa tất cả 30 đặc trưng và biến mục tiêu `is_phishing`.
Do kích thước 31×31 khá lớn, ta sẽ **không ghi chú số** mà chỉ dùng màu sắc.

In [ ]:
# === Ma trận tương quan toàn bộ ===
# Ghép features + target
df_corr = pd.concat([X, y], axis=1)
corr_matrix = df_corr.corr()

# Vẽ heatmap
fig, ax = plt.subplots(figsize=(16, 14))

mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

cmap = sns.diverging_palette(240, 10, as_cmap=True)  # Xanh dương ↔ Đỏ

sns.heatmap(
    corr_matrix,
    mask=mask,
    cmap=cmap,
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
    linecolor="white",
    annot=False,
    cbar_kws={"shrink": 0.8, "label": "Hệ số tương quan Pearson"},
    ax=ax
)

ax.set_title("🔥 Ma Trận Tương Quan — 30 Đặc Trưng + Biến Mục Tiêu",
             fontsize=16, fontweight="bold", pad=20)
ax.tick_params(axis="x", rotation=45, labelsize=9)
ax.tick_params(axis="y", rotation=0, labelsize=9)

plt.tight_layout()
save_path = os.path.join(REPORT_DIR, "eda_correlation_matrix.png")
fig.savefig(save_path, dpi=180, bbox_inches="tight", facecolor="white")
print(f"💾 Đã lưu: {save_path}")
plt.show()

In [ ]:
# === Các cặp đặc trưng có |tương quan| > 0.5 ===
threshold = 0.5
high_corr_pairs = []

for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        val = corr_matrix.iloc[i, j]
        if abs(val) > threshold:
            high_corr_pairs.append({
                "Đặc trưng 1": corr_matrix.columns[i],
                "Đặc trưng 2": corr_matrix.columns[j],
                "Tương quan": round(val, 4)
            })

if high_corr_pairs:
    df_high_corr = pd.DataFrame(high_corr_pairs).sort_values("Tương quan", key=abs, ascending=False)
    print(f"🔍 Tìm thấy {len(df_high_corr)} cặp đặc trưng có |tương quan| > {threshold}:")
    print()
    print(df_high_corr.to_string(index=False))
else:
    print(f"✅ Không có cặp đặc trưng nào có |tương quan| > {threshold}.")

---
## 6. 🎯 Tương quan Đặc trưng — Biến mục tiêu

Đánh giá mức độ tương quan (Pearson) của từng đặc trưng với biến mục tiêu `is_phishing`.
Các đặc trưng có tương quan cao (dương hoặc âm) là ứng viên quan trọng nhất cho mô hình.

In [ ]:
# === Tương quan từng đặc trưng với is_phishing ===
target_corr = corr_matrix["is_phishing"].drop("is_phishing").sort_values(key=abs, ascending=True)

# Bảng tương quan (sắp xếp giảm dần theo |giá trị|)
print("📊 Tương quan từng đặc trưng với is_phishing (sắp xếp theo |giá trị|):")
print()
for feat, val in target_corr.sort_values(key=abs, ascending=False).items():
    direction = "🟢" if val > 0 else "🔴"
    print(f"   {direction} {feat:35s}  {val:+.4f}")

# Top 10
top10 = target_corr.sort_values(key=abs, ascending=False).head(10)
print("\n🏆 Top 10 đặc trưng quan trọng nhất (theo |tương quan| với mục tiêu):")
for rank, (feat, val) in enumerate(top10.items(), 1):
    print(f"   {rank:2d}. {feat:35s}  {val:+.4f}")

In [ ]:
# === Biểu đồ thanh ngang: Tương quan đặc trưng — Biến mục tiêu ===
fig, ax = plt.subplots(figsize=(12, 10))

# Màu sắc: xanh lá = dương (tăng is_phishing), đỏ = âm
colors = ["#2ecc71" if v > 0 else "#e74c3c" for v in target_corr.values]

bars = ax.barh(target_corr.index, target_corr.values, color=colors,
               edgecolor="white", linewidth=0.8, height=0.7)

# Đường tham chiếu
ax.axvline(x=0, color="#2c3e50", linewidth=1.2)
ax.axvline(x=0.3, color="#95a5a6", linewidth=0.8, linestyle="--", alpha=0.7)
ax.axvline(x=-0.3, color="#95a5a6", linewidth=0.8, linestyle="--", alpha=0.7)

ax.set_title("🎯 Tương Quan Pearson: Đặc Trưng → is_phishing",
             fontsize=16, fontweight="bold", pad=15)
ax.set_xlabel("Hệ số tương quan Pearson", fontsize=12)
ax.set_ylabel("")
ax.tick_params(labelsize=10)

# Ghi chú giá trị trên mỗi thanh
for bar, val in zip(bars, target_corr.values):
    x_pos = val + 0.01 if val >= 0 else val - 0.01
    ha = "left" if val >= 0 else "right"
    ax.text(x_pos, bar.get_y() + bar.get_height()/2,
            f"{val:+.3f}", ha=ha, va="center", fontsize=8, fontweight="bold")

# Chú thích
ax.text(0.98, 0.02, "🟢 Dương = tăng khả năng phishing\n🔴 Âm = giảm khả năng phishing",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.5", facecolor="#ecf0f1", alpha=0.8))

plt.tight_layout()
save_path = os.path.join(REPORT_DIR, "eda_feature_target_correlation.png")
fig.savefig(save_path, dpi=180, bbox_inches="tight", facecolor="white")
print(f"💾 Đã lưu: {save_path}")
plt.show()

---
## 7. 🔬 Phân phối đặc trưng theo lớp (Feature Distribution by Class)

Với **8 đặc trưng quan trọng nhất** (theo tương quan với mục tiêu), ta phân tích sự khác biệt
trong phân phối giá trị giữa hai lớp: **hợp lệ (Legitimate)** và **lừa đảo (Phishing)**.

Nếu phân phối khác biệt rõ rệt, đặc trưng đó sẽ có giá trị phân biệt cao.

In [ ]:
# === Phân phối đặc trưng theo lớp — Top 8 quan trọng nhất ===
top8_features = target_corr.sort_values(key=abs, ascending=False).head(8).index.tolist()

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
fig.suptitle("🔬 Phân Phối Đặc Trưng Theo Lớp — Top 8 Quan Trọng Nhất",
             fontsize=18, fontweight="bold", y=1.02)

class_colors = {"Hợp lệ": "#2ecc71", "Lừa đảo": "#e74c3c"}

for idx, feat in enumerate(top8_features):
    row, col = divmod(idx, 4)
    ax = axes[row][col]
    
    # Tạo DataFrame phân tích
    temp = pd.DataFrame({feat: X[feat], "Loại": y.map({0: "Hợp lệ", 1: "Lừa đảo"})})
    
    # Tính tỷ lệ phần trăm theo lớp
    ct = pd.crosstab(temp[feat], temp["Loại"], normalize="columns") * 100
    ct = ct.reindex(columns=["Hợp lệ", "Lừa đảo"])
    
    ct.plot(kind="bar", ax=ax, color=[class_colors["Hợp lệ"], class_colors["Lừa đảo"]],
            edgecolor="white", linewidth=1.2, width=0.6)
    
    ax.set_title(feat, fontsize=11, fontweight="bold", pad=8)
    ax.set_xlabel("Giá trị", fontsize=9)
    ax.set_ylabel("Tỷ lệ (%)", fontsize=9)
    ax.tick_params(axis="x", rotation=0, labelsize=9)
    ax.tick_params(axis="y", labelsize=9)
    ax.legend(fontsize=8, loc="upper right")
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))

plt.tight_layout()
save_path = os.path.join(REPORT_DIR, "eda_feature_by_class.png")
fig.savefig(save_path, dpi=180, bbox_inches="tight", facecolor="white")
print(f"💾 Đã lưu: {save_path}")
plt.show()

### 💡 Nhận xét

- Các đặc trưng có sự khác biệt phân phối **rõ rệt** giữa hai lớp → khả năng phân biệt tốt.
- Ví dụ: `SSLfinal_State`, `URL_of_Anchor`, `web_traffic` cho thấy phân phối rất khác nhau
  giữa website hợp lệ và website lừa đảo.
- Các đặc trưng có thanh "chồng lấp" nhiều → khả năng phân biệt yếu hơn, nhưng vẫn có giá trị
  khi kết hợp với các đặc trưng khác.

---
## 8. 🔗 Các cặp đặc trưng tương quan cao (Đa cộng tuyến)

Phân tích các cặp đặc trưng có **|tương quan| > 0.5** để đánh giá mức độ đa cộng tuyến
(multicollinearity). Đa cộng tuyến cao có thể ảnh hưởng đến hiệu suất của một số mô hình
(đặc biệt là hồi quy logistic) và khiến việc diễn giải trọng số đặc trưng trở nên khó khăn.

In [ ]:
# === Phân tích đa cộng tuyến ===
# Chỉ xét tương quan giữa các features (không bao gồm target)
corr_features_only = X.corr()

high_pairs = []
for i in range(len(corr_features_only.columns)):
    for j in range(i + 1, len(corr_features_only.columns)):
        val = corr_features_only.iloc[i, j]
        if abs(val) > threshold:
            high_pairs.append({
                "Đặc trưng 1": corr_features_only.columns[i],
                "Đặc trưng 2": corr_features_only.columns[j],
                "Hệ số tương quan": round(val, 4),
                "Mức độ": "🔴 Rất cao" if abs(val) > 0.7 else "🟡 Cao"
            })

if high_pairs:
    df_pairs = pd.DataFrame(high_pairs).sort_values("Hệ số tương quan", key=abs, ascending=False)
    print(f"🔗 Tìm thấy {len(df_pairs)} cặp đặc trưng có |tương quan| > {threshold}:")
    print()
    print(df_pairs.to_string(index=False))
    print()
    print("📝 Ghi chú:")
    print("   - 🔴 Rất cao (|r| > 0.7): Nên xem xét loại bỏ một trong hai đặc trưng.")
    print("   - 🟡 Cao (0.5 < |r| ≤ 0.7): Theo dõi, có thể chấp nhận được với mô hình tree-based.")
else:
    print(f"✅ Không có cặp đặc trưng nào có |tương quan| > {threshold}.")
    print("   → Không phát hiện vấn đề đa cộng tuyến nghiêm trọng.")

### 💡 Nhận xét về Đa Cộng Tuyến

- Nếu tồn tại các cặp đặc trưng có tương quan rất cao (|r| > 0.7), chúng chứa thông tin **dư thừa**.
- Đối với mô hình **tree-based** (Random Forest, XGBoost), đa cộng tuyến thường **không ảnh hưởng** 
  đến hiệu suất dự đoán, nhưng sẽ ảnh hưởng đến feature importance.
- Đối với mô hình **tuyến tính** (Logistic Regression), nên xem xét loại bỏ hoặc kết hợp các 
  đặc trưng tương quan cao để cải thiện tính ổn định của trọng số.

---
## 9. 📝 Tổng kết & Phát hiện chính

### 📋 Tóm tắt bộ dữ liệu

| Thuộc tính | Giá trị |
|---|---|
| Nguồn | UCI Machine Learning Repository — Phishing Websites |
| Số mẫu | ~11,055 |
| Số đặc trưng | 30 |
| Kiểu đặc trưng | Phân loại (categorical): {-1, 0, 1} hoặc {-1, 1} hoặc {0, 1} |
| Biến mục tiêu | Result (gốc: -1/1) → is_phishing (chuyển đổi: 0/1) |
| Giá trị thiếu | Không có |
| Cân bằng lớp | Tương đối cân bằng |

### 🏆 Phát hiện chính

1. **Dữ liệu sạch:** Bộ dữ liệu không có giá trị thiếu, tất cả đặc trưng đều là số nguyên rời rạc.

2. **Cân bằng lớp:** Tỷ lệ phishing vs legitimate khá cân bằng → không cần kỹ thuật oversampling/undersampling.

3. **Đặc trưng phân biệt tốt:** Một số đặc trưng có tương quan mạnh với biến mục tiêu, đặc biệt:
   - Các đặc trưng liên quan đến SSL/HTTPS, URL structure, và domain reputation
   - Những đặc trưng này nên được ưu tiên trong feature selection

4. **Đa cộng tuyến:** Cần theo dõi các cặp đặc trưng có tương quan cao khi sử dụng mô hình tuyến tính.

5. **Đặc trưng ít hữu ích:** Một số đặc trưng có tương quan gần 0 với mục tiêu → có thể xem xét loại bỏ.

### 🎯 Khuyến nghị cho giai đoạn tiếp theo

1. **Feature Selection:** Xem xét loại bỏ các đặc trưng có tương quan yếu (|r| < 0.05) với mục tiêu.
2. **Mô hình ưu tiên:** Bộ dữ liệu phù hợp với các mô hình tree-based (Random Forest, XGBoost, LightGBM) do tính chất phân loại của đặc trưng.
3. **Cross-validation:** Sử dụng Stratified K-Fold để đảm bảo tỷ lệ lớp được duy trì trong mỗi fold.
4. **Đánh giá:** Sử dụng F1-score, Precision, Recall, và AUC-ROC bên cạnh Accuracy vì bài toán security yêu cầu cân bằng giữa phát hiện đúng và cảnh báo sai.